In [6]:
from functions import get_model_tokenizer
from sklearn.preprocessing import LabelEncoder
from get_dataset import download_prepare_dataset, dataset
from sklearn.model_selection import train_test_split
import numpy as np
import torch
from sklearn.svm import SVC
from sklearn.metrics import classification_report

In [7]:
model, tokenizer = get_model_tokenizer()

Some weights of the model checkpoint at lanwuwei/GigaBERT-v3-Arabic-and-English were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [8]:
le= LabelEncoder()
device = 'mps'
data_df = download_prepare_dataset().sample(1000)
print("Data Distribution: ")
print(data_df['category'].value_counts())
data_df['category'] = le.fit_transform(data_df['category'])
print()
print("Classes are: ", len(le.classes_) , " which are: " ,  le.classes_)

Data Distribution: 
category
Religion    160
Medical     158
Politics    141
Culture     140
Sports      136
Tech        136
Finance     129
Name: count, dtype: int64

Classes are:  7  which are:  ['Culture' 'Finance' 'Medical' 'Politics' 'Religion' 'Sports' 'Tech']


In [9]:
X_train, X_test, y_train, y_test = train_test_split(data_df["text"], data_df["category"], test_size=0.3)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5)
print("Training set is: ", X_train.shape[0], " rows which is ", round(X_train.shape[0]/data_df.shape[0],4)*100, "%")
print("Validation set is: ",X_val.shape[0], " rows which is ", round(X_val.shape[0]/data_df.shape[0],4)*100, "%")
print("Testing set is: ",X_test.shape[0], " rows which is ", round(X_test.shape[0]/data_df.shape[0],4)*100, "%")

Training set is:  700  rows which is  70.0 %
Validation set is:  150  rows which is  15.0 %
Testing set is:  150  rows which is  15.0 %


In [10]:
model.to(device)
with torch.no_grad():
    X_train = model(**tokenizer(X_train.tolist(), max_length = 150, truncation = True, padding = 'max_length', return_tensors = 'pt').to(device)).last_hidden_state[:, 0, :]
    X_val = model(**tokenizer(X_val.tolist(), max_length = 150, truncation = True, padding = 'max_length', return_tensors = 'pt').to(device)).last_hidden_state[:, 0, :]
    X_test = model(**tokenizer(X_test.tolist(), max_length = 150, truncation = True, padding = 'max_length', return_tensors = 'pt').to(device)).last_hidden_state[:, 0, :]

In [14]:
X_train = X_train.cpu().numpy()
X_val = X_val.cpu().numpy()
X_test = X_test.cpu().numpy()

In [15]:
clf = SVC(kernel = 'linear')
clf.fit(X_train, y_train)

SVC(kernel='linear')

In [16]:
predictions = clf.predict(X_test)

In [17]:
print(classification_report(y_test, predictions, target_names=le.classes_))

              precision    recall  f1-score   support

     Culture       0.90      0.60      0.72        15
     Finance       1.00      0.82      0.90        22
     Medical       0.73      1.00      0.85        22
    Politics       0.80      1.00      0.89        28
    Religion       0.88      0.85      0.87        27
      Sports       1.00      0.73      0.84        22
        Tech       0.80      0.86      0.83        14

    accuracy                           0.85       150
   macro avg       0.87      0.84      0.84       150
weighted avg       0.87      0.85      0.85       150



In [18]:
class_report= classification_report(y_test, predictions, target_names=le.classes_)

In [ ]:
print(class_report)
report_columns =  ["Class", "Precision", "Recall", "F1-score", "Support"]

report_table = []
for line in class_report[2:(len(le.classes_)+2)]:
    report_table.append(line.split())


              precision    recall  f1-score   support

     Culture       0.90      0.60      0.72        15
     Finance       1.00      0.82      0.90        22
     Medical       0.73      1.00      0.85        22
    Politics       0.80      1.00      0.89        28
    Religion       0.88      0.85      0.87        27
      Sports       1.00      0.73      0.84        22
        Tech       0.80      0.86      0.83        14

    accuracy                           0.85       150
   macro avg       0.87      0.84      0.84       150
weighted avg       0.87      0.85      0.85       150



In [24]:
report_table

[['Culture', '0.90', '0.60', '0.72', '15'],
 ['Finance', '1.00', '0.82', '0.90', '22'],
 ['Medical', '0.73', '1.00', '0.85', '22'],
 ['Politics', '0.80', '1.00', '0.89', '28'],
 ['Religion', '0.88', '0.85', '0.87', '27'],
 ['Sports', '1.00', '0.73', '0.84', '22'],
 ['Tech', '0.80', '0.86', '0.83', '14']]

In [ ]:
import wandb
run = wandb.init(entity="omaratef3221", project="Fake News Project - RBF", 
                         name = "Fake News Normal Head",
                         config={
                             "learning_rate": 1e-5, "architecture": 
                             "BERT with Dense Classification Head", 
                             "dataset": "SANAD: Single-Label Arabic News Articles Dataset",
                             "epochs": 10, 
                             "batch_size": 32})

run.log({
    "Classification Report": wandb.Table(data=report_table, columns=report_columns),
    })

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


wandb: WARNING Fatal error while uploading data. Some run data will not be synced, but it will still be written to disk. Use `wandb sync` at the end of the run to try uploading.
